# Import packages

In [1]:
import os
import pandas as pd
import json
from dateutil.parser import parse
from datetime import datetime
import time

from tqdm import tqdm
import requests

import yfinance as yf

# Set global variables

In [2]:
# Define data directory
data_dir = os.path.join("..", "data")

# Define sector ETFs
sector_etfs        = {"Energy": "XLE", "Materials": "XLB", "Industrials": "XLI", "Healthcare": "XLV",
                      "Financials": "XLF", "Technology": "XLK", "Telecommunication Services": "XLC",
                      "Utilities": "XLU", "Real Estate": "XLRE", "Clean Energy": "ICLN", "Semiconductors": "SOXX",
                      "Renewables": "TAN", "Cloud Computing": "SKYY", "Genomics": "ARKG",
                     }

# # Extract sector concept ID from OpenAlex
# def get_concept_id(concept_name):
#     url = "https://api.openalex.org/concepts"
#     params = {"search": concept_name}
#     response = requests.get(url, params=params)
#     if response.status_code == 200:
#         results = response.json().get('results', [])
#         if results:
#             # Return the ID of the top result
#             return results[0]['id']
#     return None

# sector_concept_ids = {}
# for sector in sector_etfs:
#     concept_id = get_concept_id(sector)
#     if concept_id:
#         sector_concept_ids[sector] = concept_id.rsplit('/', 1)[-1]
#     else:
#         print(f"Concept ID not found for sector: {sector}")
# print(sector_concept_ids)

sector_concept_ids = {'Energy': 'C188573790', 'Materials': 'C192562407', 'Industrials': 'C40700', 'Healthcare': 'C2779951463',
                      'Financials': 'C106159729', 'Technology': 'C16443162', 'Telecommunication Services': 'C195358072',
                      'Utilities': 'C205706631', 'Real Estate': 'C82279013', 'Clean Energy': 'C2988459927', 'Semiconductors': 'C43214815',
                      'Renewables': 'C188573790', 'Cloud Computing': 'C79974875', 'Genomics': 'C189206191'
                     }

# Fetch ETFS

In [3]:
def fetch_sector_data(sector_etfs, period = "1y", interval = "1wk"):
    """
    Fetches time series data for a dictionary of sector ETFs and returns it in long format:
    Date, Industry, Open, High, Low, Close, Volume.

    Parameters:
        sector_etfs (dict): Dictionary mapping sector names to ETF tickers.
        period (str): Period of data to fetch (e.g., '1y', '6mo').
        interval (str): Data interval (e.g., '1wk', '1d').

    Returns:
        pd.DataFrame: Long-form DataFrame with columns: Date, Industry, Open, High, Low, Close, Volume.
    """
    tickers = list(sector_etfs.values())

    # Download historical price data
    df = yf.download(tickers, period = period, interval = interval, progress = False)

    # Convert from wide to long format
    df_long = df.stack(level = 1, future_stack = True).reset_index()

    # Rename the column holding tickers
    df_long = df_long.rename(columns = {'Ticker': 'Industry'})

    # Map tickers back to sector names
    inv_map             = {v: k for k, v in sector_etfs.items()}
    df_long['Industry'] = df_long['Industry'].map(inv_map)

    return df_long

# Fetch Publications

In [4]:
def fetch_publication_data(concept_ids, from_date = "2025-01-01", to_date = "2025-12-31", max_per_sector = 50000, period = 'D'):
    """
    Fetches time series data for a dictionary of publications and returns it in long format:
    date, industry, publications, citations

    Parameters:
        concept_ids (dict): Dictionary mapping sector names to concept IDs.
        from_date (str): Start period of data retrieval.
        to_date (str): End period of data retrieval.
        max_per_sector (int): maximum number of publications to retrieve
        period: period of retrieval - 'D' (day), 'W' (week), 'M' (month)

    Returns:
        pd.DataFrame: Long-form DataFrame with columns: date, industry, publications, citations
    """
    
    base_url = "https://api.openalex.org/works"
    stats    = []

    for sector, concept_id in concept_ids.items():
        print(f"Fetching for {sector}")
        cursor = "*"
        total  = 0

        while True:
            filter_string = f"concepts.id:{concept_id},from_publication_date:{from_date},to_publication_date:{to_date}"
            params        = {"filter": filter_string,
                             "per-page": 200,
                             "cursor": cursor
                            }

            response = requests.get(base_url, params = params)
            data     = response.json()
            works    = data.get("results", [])
            cursor   = data.get("meta", {}).get("next_cursor")

            for work in works:
                stats.append({"sector": sector,
                              "date": work.get("publication_date"),
                              "citations": work.get("cited_by_count", 0)
                            })

            total += len(works)
            if not cursor or total >= max_per_sector:
                break

            time.sleep(1)

    df                 = pd.DataFrame(stats)
    df["date"]         = pd.to_datetime(df["date"])
    df["publications"] = 1
    if period == 'D':
        df             = df.groupby(["sector", "date"]).agg(publications = ("publications", "sum"),
                                                            citations = ("citations", "sum")).reset_index()
    elif period == 'M':
        df = df.groupby(["sector", df["date"].dt.to_period(period)]).agg(publications = ("publications", "sum"),
                                                                         citations = ("citations", "sum")
                                                                        ).reset_index()
    else:
        df = df.groupby(["sector", df["date"].dt.to_period(period)]).agg(publications = ("publications", "sum"),
                                                                         citations = ("citations", "sum")
                                                                        ).reset_index()
        
    df["date"]         = df["date"].astype(str)
    
    return df

# Save Data

### Stock Data

In [51]:
daily_stocks   = fetch_sector_data(sector_etfs, interval = "1d")
weekly_stocks  = fetch_sector_data(sector_etfs, interval = "1wk")
monthly_stocks = fetch_sector_data(sector_etfs, interval = "1mo")

# Save DataFrames
daily_stocks.to_parquet(os.path.join(data_dir, "daily_stocks.parquet"))
weekly_stocks.to_parquet(os.path.join(data_dir, "weekly_stocks.parquet"))
monthly_stocks.to_parquet(os.path.join(data_dir, "monthly_stocks.parquet"))

### Publication Data

In [5]:
df_daily   = fetch_publication_data(sector_concept_ids, from_date = "2025-01-01", to_date = "2025-04-30", period = "D")
df_weekly  = fetch_publication_data(sector_concept_ids, from_date = "2025-01-01", to_date = "2025-04-30", period = "W-MON")
df_monthly = fetch_publication_data(sector_concept_ids, from_date = "2025-01-01", to_date = "2025-04-30", period = "M")

df_daily.to_parquet(os.path.join(data_dir, "daily_publications.parquet"))
df_weekly.to_parquet(os.path.join(data_dir, "weekly_publications.parquet"))
df_monthly.to_parquet(os.path.join(data_dir, "monthly_publications.parquet"))

Fetching for Energy
Fetching for Materials
Fetching for Industrials
Fetching for Healthcare
Fetching for Financials
Fetching for Technology
Fetching for Telecommunication Services
Fetching for Utilities
Fetching for Real Estate
Fetching for Clean Energy
Fetching for Semiconductors
Fetching for Renewables
Fetching for Cloud Computing
Fetching for Genomics
